# MineToday 2026 — AI Course Advisor: Pemeringkat Modul Pembelajaran
**Kompetisi:** IT Today IPB 2026 · **Tim:** cupuu · **Metrik:** NDCG@5 · **CV:** 0.66555

---

## Pipeline Overview

| Bab | Tahap | Deskripsi |
|-----|-------|----------|
| 01 | **Setup** | Konstanta global, impor, dan penguncian benih keacakan |
| 02 | **Business Understanding** | Definisi masalah, struktur label, dan implikasi NDCG@5 |
| 03 | **Data Understanding** | Lima berkas CSV, sinyal percakapan, cakupan asesmen |
| 04 | **Data Preparation** | Normalisasi asesmen: kata bilangan, pencilan, nilai kosong |
| 05 | **Feature Engineering & Split** | Empat keluarga fitur; *RepeatedKFold* 5×3 = 15 lipatan |
| 06 | **Modeling** | *LightGBM* regresi + *LambdaRank*, *bagging* lima benih |
| 07 | **Evaluation** | Optuna 50 percobaan, OOF 15 lipatan, ambang 2 SE |
| 08 | **Save & Load Model** | Bundel *cloudpickle* mandiri `ChatIntentSkillGapRanker` |
| 09 | **Prediction** | Inferensi dari CSV mentah, verifikasi SHA-256 |


In [1]:
SEED = 42                 # benih tunggal yang mengatur seluruh keacakan notebook
N_SEEDS = 3               # jumlah pengulangan RepeatedKFold; 3 x 5 menghasilkan 15 lipatan
N_FOLDS = 5               # jumlah lipatan per pengulangan
N_TRIALS = 50             # jumlah percobaan Optuna pada lipatan penyetelan
N_ESTIMATORS = 900        # jumlah pohon LightGBM awal sebelum disetel
LEARNING_RATE = 0.03      # laju belajar LightGBM awal sebelum disetel
NUM_LEAVES = 63           # lebar pohon LightGBM awal sebelum disetel
RECENCY_DECAY = 0.65      # faktor peluruhan bobot resensi; tiap pesan mundur dikali 0.65
PROJECT_ROOT = ".."       # akar repositori relatif terhadap direktori eksperimen
EXP_DIR = "history/exp-027_exp-027-blend-reg-rank"   # direktori keluaran untuk model dan submission


In [2]:
import os                                      # dipakai hanya untuk menetapkan PYTHONHASHSEED

os.environ["PYTHONHASHSEED"] = "42"            # kunci hash string Python sebelum modul lain diimpor

import hashlib                                 # SHA-256 untuk verifikasi identitas submission
import json                                    # baca kolom asesmen berformat JSON, tulis ringkasan
import random                                  # generator acak baku Python, ikut dibenihkan
import re                                      # tokenisasi teks percakapan berbasis regex
import sys                                     # sisipkan akar proyek ke jalur impor
from pathlib import Path                       # penanganan path lintas sistem operasi

import numpy as np                             # aljabar matriks untuk seluruh blok fitur
import pandas as pd                            # pemuatan CSV dan perakitan bingkai fitur

random.seed(SEED)                              # benihkan generator acak Python
np.random.seed(SEED)                           # benihkan generator acak global NumPy
rng = np.random.default_rng(SEED)              # generator NumPy modern, dibenihkan terpisah
N_THREADS = 4                                  # jumlah utas eksplisit; n_jobs=-1 merusak determinisme

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()    # ubah akar proyek menjadi path absolut
EXP_DIR = Path(EXP_DIR).resolve()             # ubah direktori eksperimen menjadi path absolut
RAW = PROJECT_ROOT / "data" / "raw"            # lokasi lima berkas CSV mentah kompetisi
sys.path.insert(0, str(PROJECT_ROOT))          # agar modul metrics.py di akar dapat diimpor

from metrics import ndcg_at5, ndcg_full        # metrik resmi (@5) dan metrik pembanding (penuh)

pd.set_option("display.width", 220)            # lebarkan cetakan tabel agar kolom tidak terpotong
pd.set_option("display.max_columns", 60)       # tampilkan seluruh kolom bingkai fitur saat dicetak
print(f"root={PROJECT_ROOT}")                  # konfirmasi akar proyek yang benar-benar dipakai


root=D:\2. College\1. Self-development\0. Lomba\35. IT Today IPB 2026


---

## 1. Business Understanding

Platform Intelligo membutuhkan sistem yang dapat memeringkat 17 modul pembelajaran dari yang
paling relevan untuk dipelajari berikutnya oleh setiap pengguna. Masukan yang tersedia ada tiga
jenis: asesmen mandiri sepuluh pertanyaan, riwayat percakapan berbahasa Indonesia, dan katalog
modul yang memuat silabus serta jenjang prasyarat.

Metrik resmi kompetisi adalah **NDCG@5**, dan pilihan metrik itu mengubah bentuk persoalannya
secara mendasar. Hanya lima peringkat pertama yang dihitung, dan bobot DCG pada posisi 1 sampai
5 masing-masing adalah 1.00, 0.63, 0.50, 0.43, dan 0.39 — sehingga posisi pertama saja
memikul sekitar 26% dari seluruh massa skor yang dapat diperoleh.

Dari eksplorasi data latih, kami menemukan struktur yang menentukan pendekatan: tepat satu
modul per pengguna bernilai persis 1.0, berlaku pada 4.000 dari 4.000 pengguna tanpa
pengecualian. Nilai selebihnya mengikuti tangga tetap 1.00, 0.85, 0.70, 0.55, 0.40, 0.25
dengan guncangan hanya ±0.029, sehingga nilai relevansi tidak membawa informasi apa pun di
luar urutan relatifnya.

> **Insight:** menempatkan modul bernilai 1.0 pada posisi pertama dan membiarkan sisanya
> diurutkan berdasarkan popularitas saja sudah menghasilkan NDCG@5 = 0.64086, dibandingkan
> 0.39713 untuk *baseline* popularitas murni. *Retrieval* modul teratas bernilai 0.241 poin;
> seluruh perbaikan *ordering* posisi 2–5 hanya menyumbang selebihnya. Kapasitas model yang
> dihabiskan untuk merapikan ekor daftar karenanya terbuang percuma.


---

## 2. Data Understanding

Lima berkas CSV menyusun seluruh data kompetisi: `train_relevance.csv` berisi 4.000 pengguna
latih dengan label *wide* 17 modul, `test.csv` berisi 1.000 pengguna uji tanpa label,
`user_assessments.csv` berisi satu kolom JSON per pengguna yang memuat skor asesmen mandiri,
`chat_history.csv` berisi riwayat percakapan dengan satu baris per pesan, dan
`modules_catalog.csv` berisi metadata modul beserta silabus dan jenjang prasyarat.

Percakapan menyimpan sinyal yang lebih besar daripada asesmen karena perbedaan jenis informasi
yang dikandungnya: asesmen menyatakan apa yang sudah dikuasai pengguna, sedangkan percakapan
menyatakan apa yang ia inginkan. Sekitar 3% pengguna uji sama sekali tidak memiliki percakapan,
dan pengguna tersebut harus tetap memperoleh prediksi yang masuk akal dari asesmen saja.


In [3]:
train = pd.read_csv(RAW / "train_relevance.csv")      # 4.000 pengguna latih, label wide 17 modul
test = pd.read_csv(RAW / "test.csv")                  # 1.000 pengguna uji, hanya kolom user_id
assess = pd.read_csv(RAW / "user_assessments.csv")    # asesmen mandiri, satu kolom JSON per pengguna
chat = pd.read_csv(RAW / "chat_history.csv")          # riwayat percakapan, satu baris per pesan
catalog = pd.read_csv(RAW / "modules_catalog.csv")    # katalog modul beserta silabus dan prasyarat

MODULES = [c for c in train.columns if c != "user_id"]              # 17 id modul, urutan kolom label
Y = train[MODULES].to_numpy()                                       # matriks relevansi 4000 x 17
QCOLS = list(json.loads(assess.assessment_result.iloc[0]).keys())   # 10 nama pertanyaan asesmen

chat["timestamp"] = pd.to_datetime(chat["timestamp"])                       # ubah stempel waktu ke datetime
chat = chat.sort_values(["user_id", "timestamp"]).reset_index(drop=True)    # urutkan kronologis per pengguna
print(f"pesan: {len(chat)}  pengguna dengan chat: {chat.user_id.nunique()}")
print(f"train tanpa chat: {len(set(train.user_id) - set(chat.user_id))}")
print(f"modul: {MODULES}")


pesan: 14573  pengguna dengan chat: 4940
train tanpa chat: 117
modul: ['M_001', 'M_002', 'M_003', 'M_004', 'M_005', 'M_006', 'M_007', 'M_008', 'M_009', 'M_010', 'M_011', 'M_012', 'M_013', 'M_014', 'M_015', 'M_016', 'M_017']


In [4]:
example = chat[chat.user_id == "U_0002"][["timestamp", "user_chat_text"]]   # satu pengguna sebagai contoh kasus
print(example.to_string(index=False))                                       # cetak utuh tanpa indeks baris


          timestamp                                                                       user_chat_text
2026-01-12 10:37:59               Min aku mau fokus jadi data Analyst aja deh mau kuasain Excel sama SQL
2026-03-12 10:37:59         Eh min gajadi deh bosku nyuruh pindah haluan bikin AI Ada kelas GenAI nggak?
2026-04-01 23:53:55 Sori nanya mulu ya min, tapi kelas yg cover data cleaning ada silabus lengkapnya ga?
2026-04-04 10:18:29  Sori nanya mulu ya min tapi kelas belajar pyton dari nol itu berapa lama durasinya?


> **Insight:** percakapan pengguna dapat mengandung niat yang saling bertentangan di waktu berbeda.
> Agregat datar atas seluruh pesan akan mencampur sinyal yang meniadakan satu sama lain.
> Pembobotan resensi karena itu bukan penyetelan halus melainkan syarat agar representasi teks
> yang dibangun benar. Temuan ini memotivasi `RECENCY_DECAY = 0.65` sehingga tiap langkah
> mundur satu pesan memangkas bobot menjadi 0.65 kali, dan `text_sim_last` disediakan sebagai
> kolom terpisah agar model dapat membedakan niat terakhir dari riwayat keseluruhan.


---

## 3. Data Preparation

Kolom asesmen mengandung tiga jenis ketidakkonsistenan yang kami tangani sebelum fitur dapat dirakit.

- **Kata bilangan bahasa Indonesia** ("Nol", "Satu", "Empat") bercampur dengan angka, sehingga
  `pd.to_numeric` saja menghasilkan NaN pada baris yang sebenarnya sah.
- **Nilai di luar rentang** seperti -3 dan 99 muncul pada skala yang seharusnya 0 sampai 5,
  dan bila dibiarkan akan menggeser rata-rata pengguna secara ekstrem.
- **Sel kosong** tersisa sekitar 383 setelah koersi, yang diisi dengan median pengguna latih
  tanpa membocorkan informasi dari data uji.

Pentingnya pembersihan ini terukur: korelasi absolut maksimum antara satu pertanyaan asesmen
dan satu modul naik dari 0.192 pada data mentah menjadi **0.416** setelah kata bilangan
dipetakan kembali ke bilangan bulat dan pencilan dijepit. Pembacaan awal bahwa asesmen bersinyal
lemah ternyata berasal dari format yang belum dinormalisasi, bukan dari sifat datanya.


In [5]:
WORD_TO_NUM = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5}   # peta kata bilangan Indonesia


def coerce_assessment(assess_df, qcols):
    '''Ubah kolom JSON asesmen menjadi bingkai numerik bersih berskala 0-5.'''
    parsed = pd.DataFrame([json.loads(s) for s in assess_df.assessment_result])   # bongkar JSON jadi kolom
    parsed.insert(0, "user_id", assess_df.user_id.values)                         # kembalikan kunci pengguna
    for c in qcols:                                                               # tangani tiap pertanyaan terpisah
        as_word = parsed[c].map(                                                  # jalur pertama: baca sebagai kata bilangan
            lambda v: WORD_TO_NUM.get(v.strip().lower()) if isinstance(v, str) else None)
        as_num = pd.to_numeric(parsed[c], errors="coerce")                        # jalur kedua: baca sebagai angka
        parsed[c] = pd.to_numeric(as_num.where(as_num.notna(), as_word),          # angka diutamakan, kata jadi cadangan
                                  errors="coerce").clip(0, 5)                     # jepit pencilan -3 dan 99 ke rentang sah
    return parsed[["user_id"] + list(qcols)]                                      # kembalikan hanya kolom yang dipakai


A = coerce_assessment(assess, QCOLS)                                  # asesmen bersih untuk seluruh pengguna
Q_MEDIAN = A.loc[A.user_id.isin(train.user_id), QCOLS].median()       # median DIHITUNG HANYA DARI PENGGUNA LATIH
A[QCOLS] = A[QCOLS].fillna(Q_MEDIAN)                                  # isi sel kosong tanpa membocorkan data uji

MODULE_DOCS = {r.module_id: f"{r.module_name} {r.description_and_syllabus}"   # satu dokumen teks per modul
               for r in catalog.itertuples()}
CHAT_BY_USER = {uid: grp.user_chat_text.tolist()                              # daftar pesan kronologis per pengguna
                for uid, grp in chat.groupby("user_id", sort=True)}
print(f"dokumen modul: {len(MODULE_DOCS)}  contoh: {MODULE_DOCS['M_012'][:110]}")


dokumen modul: 17  contoh: Generative AI Implementasi Large Language Models (LLM), arsitektur RAG (Retrieval-Augmented Generation), dan f


---

## 4. Feature Engineering & Split

Setiap pasangan pengguna-modul memperoleh 32 kolom yang dirakit dari empat keluarga fitur,
dan tiap keluarga menjawab pertanyaan yang berbeda tentang kesesuaian pasangan itu.

- **Kesenjangan keahlian** (`gap`, `abs_gap`, `target_score`, `prereq_min`, `prereq_mean`,
  `eligible`) menjawab apakah modul berada di zona yang tepat bagi pengguna. Relevansi
  mengikuti kurva U terbalik terhadap kesenjangan: modul fondasi diminati pemula, modul lanjutan
  diminati pengguna berpengalaman. `abs_gap` disediakan berdampingan dengan `gap` bertanda agar
  pohon dapat memotong kedua sisi kurva.
- **Kemiripan teks** (`text_sim`, `text_sim_last`, `sim_rank`, `alias_hit`) menjawab apakah
  pengguna pernah menyebut hal yang dibahas modul. Kosinus TF-IDF riwayat berbobot resensi dan
  pencocokan alias literal memberi dua cara berbeda mendeteksi niat yang sama.
- **Afinitas terpelajar** (`aff_mean`, `aff_max`, `aff_last`, `ridge_text`) menjawab modul apa
  yang secara empiris menyertai istilah tertentu pada pengguna lain. Ini satu-satunya keluarga
  yang diturunkan dari label, sehingga kami pelajari hanya di dalam *fold* training.
- **Konteks pengguna** (`user_mean`, `user_std`, `pop_prior`, `module_idx`, `level_ord`,
  `q0`–`q9`) memberi latar agar dua pasangan dengan kesenjangan sama tetap dapat dibedakan.

*RepeatedKFold* 5×3 = 15 lipatan digunakan karena varians antar lipatan tinggi — delta yang
lebih kecil dari dua *standard error* ≈ 0.0033 bukan sinyal nyata dan tidak layak disubmit.


In [6]:
ALIASES = {                                                        # istilah literal khas tiap modul
    "M_001": ["excel", "spreadsheet", "pivot", "vlookup"],
    "M_002": ["python", "pyton", "phyton", "pandas", "numpy"],
    "M_003": ["sql", "query", "database", "join"],
    "M_004": ["scrap", "scraping", "beautifulsoup", "selenium"],
    "M_005": ["git", "github", "version control"],
    "M_006": ["statistik", "statistic", "probabilitas"],
    "M_007": ["eda", "eksplor", "data cleaning", "cleaning", "insight"],
    "M_008": ["dashboard", "tableau", "power bi", "powerbi", "visualisasi"],
    "M_009": ["machine learning", "ml ", "prediksi", "model"],
    "M_010": ["computer vision", "citra", "gambar", "cnn"],
    "M_011": ["nlp", "natural language", "teks"],
    "M_012": ["genai", "generative", "llm", "gpt", "gen ai"],
    "M_013": ["prompt"],
    "M_014": ["automation", "otomasi", "workflow", "agent"],
    "M_015": ["no code", "nocode", "tanpa coding", "tanpa ngoding"],
    "M_016": ["mlops", "deploy", "produksi", "production"],
    "M_017": ["karir", "karier", "portofolio", "portfolio", "interview", "lamaran"],
}

MIN_TERM_USERS = 25       # istilah harus muncul pada >= 25 pengguna training
SHRINK_K = 40.0           # penyusutan n/(n+k); menahan lift ekstrem dari istilah langka
TOKEN_RE = r"[a-z]{3,}"   # token huruf kecil minimal 3 karakter


def tokenize(text):
    '''Himpunan token unik sebuah teks; keunikan membuat pengulangan kata tidak menambah bobot.'''
    return set(re.findall(TOKEN_RE, str(text).lower()))


def fit_term_affinity(user_ids, chat_by_user, Y_train, modules):
    '''Afinitas istilah->modul dipelajari dari label; wajib dipanggil hanya dengan pengguna fold training.'''
    docs = [tokenize(" ".join(chat_by_user.get(u, []))) for u in user_ids]
    counts = {}
    for d in docs:
        for t in d:
            counts[t] = counts.get(t, 0) + 1
    vocab = sorted(t for t, c in counts.items() if c >= MIN_TERM_USERS)
    index = {t: i for i, t in enumerate(vocab)}

    total = np.zeros((len(vocab), len(modules)))
    seen = np.zeros(len(vocab))
    for row, d in enumerate(docs):
        hits = [index[t] for t in d if t in index]
        if not hits:
            continue
        total[hits] += Y_train[row]                                    # tambahkan vektor relevansi ke tiap istilah
        seen[hits] += 1

    global_mean = Y_train.mean(axis=0)
    term_mean = np.divide(total, seen[:, None], out=np.zeros_like(total), where=seen[:, None] > 0)
    shrink = (seen / (seen + SHRINK_K))[:, None]                       # istilah jarang ditarik ke nol
    return {"vocab": index, "affinity": (term_mean - global_mean) * shrink}


def affinity_features(user_ids, chat_by_user, model_aff, modules, decay):
    '''(mean, max, pesan-terakhir) afinitas per pasangan pengguna-modul.'''
    index, aff = model_aff["vocab"], model_aff["affinity"]
    n_users, n_modules = len(user_ids), len(modules)
    a_mean = np.zeros((n_users, n_modules))
    a_max = np.zeros((n_users, n_modules))
    a_last = np.zeros((n_users, n_modules))
    for i, uid in enumerate(user_ids):
        messages = chat_by_user.get(uid) or []
        if not messages:
            continue
        hits = [index[t] for t in tokenize(" ".join(messages)) if t in index]
        if hits:
            block = aff[hits]
            a_mean[i] = block.mean(axis=0)
            a_max[i] = block.max(axis=0)
        last_hits = [index[t] for t in tokenize(messages[-1]) if t in index]
        if last_hits:
            a_last[i] = aff[last_hits].mean(axis=0)
    return a_mean, a_max, a_last


RIDGE_ALPHA = 3.0        # kekuatan regularisasi ridge pada matriks TF-IDF yang sangat lebar
INNER_FOLDS = 5          # jumlah lipatan dalam untuk OOF bersarang


def _ridge_matrix(user_ids, chat_by_user, vectorizer):
    '''Satu vektor TF-IDF per pengguna dari seluruh percakapannya.'''
    docs = [" ".join(chat_by_user.get(u, [])) for u in user_ids]
    return vectorizer.transform(docs)


def ridge_stack_features(tr_users, va_users, chat_by_user, Y_tr, vectorizer, seed):
    '''Prediksi relevansi dari teks saja via ridge per modul, dengan OOF bersarang.'''
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import KFold

    X_tr = _ridge_matrix(tr_users, chat_by_user, vectorizer)
    X_va = _ridge_matrix(va_users, chat_by_user, vectorizer)

    oof = np.zeros((len(tr_users), Y_tr.shape[1]))
    inner = KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
    for in_tr, in_va in inner.split(np.arange(len(tr_users))):
        model = Ridge(alpha=RIDGE_ALPHA, random_state=seed)
        model.fit(X_tr[in_tr], Y_tr[in_tr])
        oof[in_va] = model.predict(X_tr[in_va])                        # prediksi out-of-fold

    full = Ridge(alpha=RIDGE_ALPHA, random_state=seed)
    full.fit(X_tr, Y_tr)                                               # ridge final atas seluruh pengguna training
    return oof, full.predict(X_va), full


def recency_weights(n, decay):
    '''Bobot naik ke pesan terakhir: w_i = decay^(n-1-i), dinormalisasi.'''
    w = np.array([decay ** (n - 1 - i) for i in range(n)], dtype=float)
    return w / w.sum()


def alias_scores(messages, modules, aliases, decay):
    '''Kemunculan istilah khas modul dibobot resensi. Mengembalikan (n_modules,).'''
    out = np.zeros(len(modules))
    if not messages:
        return out
    w = recency_weights(len(messages), decay)
    lowered = [m.lower() for m in messages]
    for j, module in enumerate(modules):
        for i, text in enumerate(lowered):
            if any(term in text for term in aliases[module]):
                out[j] += w[i]                                         # tambahkan bobot resensi pesan yang cocok
    return out


In [7]:
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize


def make_vectorizer():
    '''Gabungan TF-IDF kata dan karakter; bigram kata menangkap frasa, n-gram karakter menahan salah eja.'''
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                                 sublinear_tf=True, lowercase=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                 sublinear_tf=True, lowercase=True)),
    ])


def text_features(user_ids, vectorizer, module_matrix, chat_by_user, modules, decay):
    '''Hitung (text_sim, text_sim_last, alias, has_chat, n_msg) untuk tiap pengguna.'''
    n_users, n_modules = len(user_ids), len(modules)
    sim = np.zeros((n_users, n_modules))
    sim_last = np.zeros((n_users, n_modules))
    alias = np.zeros((n_users, n_modules))
    has_chat = np.zeros(n_users)
    n_msg = np.zeros(n_users)

    flat, spans = [], []
    for uid in user_ids:
        messages = chat_by_user.get(uid) or []
        spans.append((len(flat), len(messages)))
        flat.extend(messages)
    if not flat:
        return sim, sim_last, alias, has_chat, n_msg

    vecs = normalize(vectorizer.transform(flat))                       # vektorkan seluruh pesan sekali, normalisasi L2
    for i, (start, count) in enumerate(spans):
        if count == 0:
            continue
        has_chat[i] = 1.0
        n_msg[i] = count
        w = recency_weights(count, decay)
        block = vecs[start:start + count]
        weighted = normalize(sp.csr_matrix(w.reshape(1, -1)) @ block)
        sim[i] = (weighted @ module_matrix.T).ravel()
        sim_last[i] = (block[count - 1] @ module_matrix.T).ravel()
        alias[i] = alias_scores(flat[start:start + count], modules, ALIASES, decay)
    return sim, sim_last, alias, has_chat, n_msg


In [8]:
TARGET_SKILL = {"M_001": -1, "M_002": 0, "M_003": 1, "M_004": 0, "M_005": -1, "M_006": 2,
                "M_007": 3, "M_008": 1, "M_009": 4, "M_010": 6, "M_011": 6, "M_012": 7,
                "M_013": 7, "M_014": 7, "M_015": 7, "M_016": 4, "M_017": 8}
PREREQ_SKILL = {"M_001": [], "M_002": [], "M_003": [], "M_004": [0], "M_005": [], "M_006": [],
                "M_007": [0, 2], "M_008": [1], "M_009": [3, 2], "M_010": [4, 6], "M_011": [4],
                "M_012": [4, 6], "M_013": [], "M_014": [], "M_015": [], "M_016": [4, 0],
                "M_017": []}
LEVEL_ORD = {"Pemula": 0, "Menengah": 1, "Lanjutan": 2, "Ahli": 3, "Semua Level": 1}


def level_of(text):
    '''Ubah teks prasyarat katalog menjadi ordinal; nilai tak dikenal jatuh ke menengah.'''
    for name, ordinal in LEVEL_ORD.items():
        if str(text).startswith(name):
            return ordinal
    return 1


MODULE_LEVEL = {r.module_id: level_of(r.prerequisite_level) for r in catalog.itertuples()}


def build_features(user_ids, assess_clean, qcols, modules, pop_prior, module_level,
                   target_skill, prereq_skill, text_block=None, aff_block=None, ridge_block=None):
    '''Rakit bingkai fitur user-major: 17 baris berurutan untuk tiap pengguna.'''
    a = assess_clean.set_index("user_id").loc[list(user_ids), qcols]
    scores = a.to_numpy(dtype=float)
    n_users = len(user_ids)

    user_mean = scores.mean(axis=1)                                    # rata-rata keahlian mandiri
    user_std = scores.std(axis=1)                                      # sebaran keahlian
    user_min = scores.min(axis=1)
    user_max = scores.max(axis=1)
    n_zero = (scores == 0).sum(axis=1).astype(float)                   # penanda pemula murni
    n_five = (scores == 5).sum(axis=1).astype(float)                   # penanda percaya diri tinggi

    if text_block is None:
        z = np.zeros((n_users, len(modules)))
        sim, sim_last, alias, has_chat, n_msg = z, z, z, np.zeros(n_users), np.zeros(n_users)
    else:
        sim, sim_last, alias, has_chat, n_msg = text_block

    if aff_block is None:
        z2 = np.zeros((n_users, len(modules)))
        a_mean, a_max, a_last = z2, z2, z2
    else:
        a_mean, a_max, a_last = aff_block

    ridge_pred = (np.zeros((n_users, len(modules))) if ridge_block is None else ridge_block)

    blocks = []
    for j, module in enumerate(modules):
        tq = target_skill[module]
        target = scores[:, tq] if tq >= 0 else np.full(n_users, -1.0)
        pq = prereq_skill[module]
        if pq:
            prereq_min = scores[:, pq].min(axis=1)
            prereq_mean = scores[:, pq].mean(axis=1)
        else:
            prereq_min = np.full(n_users, 5.0)                         # anggap prasyarat terpenuhi penuh
            prereq_mean = np.full(n_users, 5.0)
        level = float(module_level[module])
        gap = level * (5.0 / 3.0) - user_mean                          # jenjang diskalakan ke rentang asesmen
        block = pd.DataFrame({
            "module_idx": np.full(n_users, j),                         # identitas modul sebagai fitur kategorikal
            "level_ord": np.full(n_users, level),
            "pop_prior": np.full(n_users, pop_prior[j]),
            "target_score": target,
            "target_minus_mean": np.where(target >= 0, target - user_mean, 0.0),
            "prereq_min": prereq_min, "prereq_mean": prereq_mean,
            "eligible": (prereq_min >= 2).astype(float),
            "gap": gap, "abs_gap": np.abs(gap),
            "user_mean": user_mean, "user_std": user_std,
            "user_min": user_min, "user_max": user_max,
            "n_zero": n_zero, "n_five": n_five,
            "text_sim": sim[:, j], "text_sim_last": sim_last[:, j],
            "alias_hit": alias[:, j], "has_chat": has_chat, "n_msg": n_msg,
            "aff_mean": a_mean[:, j], "aff_max": a_max[:, j],
            "aff_last": a_last[:, j], "ridge_text": ridge_pred[:, j],
            "sim_rank": np.zeros(n_users),                             # diisi setelah seluruh modul dirakit
        })
        blocks.append(block)

    sim_rank = (-sim).argsort(axis=1).argsort(axis=1).astype(float)   # argsort ganda menghasilkan peringkat menurun
    for j in range(len(modules)):
        blocks[j]["sim_rank"] = sim_rank[:, j]

    features = pd.concat(blocks, axis=0, ignore_index=True)
    order = np.argsort(np.tile(np.arange(n_users), len(modules)), kind="stable")
    for qi in range(len(qcols)):
        features[f"q{qi}"] = np.concatenate([scores[:, qi]] * len(modules))
    return features.iloc[order].reset_index(drop=True)


CATEGORICAL = ["module_idx"]   # satu-satunya kolom kategorikal bagi LightGBM
print("fitur siap")


fitur siap

In [9]:
from sklearn.model_selection import RepeatedKFold

cv = RepeatedKFold(n_splits=N_FOLDS, n_repeats=N_SEEDS, random_state=SEED)
splits = list(cv.split(train.user_id.to_numpy()))                      # bagi pada level PENGGUNA agar tidak bocor
print(f"jumlah split: {len(splits)}")                                  # harus 15 lipatan


jumlah split: 15


---

## 5. Modeling

*LightGBM* dipilih karena bentuk sinyalnya: relevansi bergerak mengikuti kurva U terbalik
terhadap kesenjangan keahlian, dan model linier hanya dapat memasang satu kemiringan pada
hubungan yang berbalik arah di tengah. Pohon memotong `gap` pada dua ambang berbeda dan
karenanya dapat merepresentasikan kedua sisi kurva sekaligus.

Dua *objective* dilatih dan digabung lewat *rank-averaging*:

- **Regresi MSE** (disetel Optuna): meminimalkan error nilai absolut, menangkap tingkat
  relevansi absolut yang menentukan urutan posisi 2–5.
- **LambdaRank** (parameter tetap dari exp-026): memaksimalkan NDCG secara langsung pada
  `truncation_level=5`, menajamkan posisi 1–5.

Dua *objective* yang berbeda cenderung membuat kesalahan yang berbeda; *rank-averaging*
menggabungkan keduanya dalam ruang peringkat sehingga perbedaan skala tidak memengaruhi
bobot relatifnya. Tiga mekanisme kendali varian dipasang berlapis: *bagging* lima benih,
`deterministic=True`, dan `n_jobs=N_THREADS` eksplisit.


In [10]:
import lightgbm as lgb

N_BAG = 5                                                              # banyaknya anggota bagging benih
LADDER = np.array([0.0, 0.25, 0.40, 0.55, 0.70, 0.85, 1.00])          # tangga relevansi tiap posisi


def make_grade_labels(y_float):
    '''Kuantisasi label kontinu ke grade ordinal 0-6 untuk LambdaRank.'''
    return np.argmin(np.abs(y_float[:, None] - LADDER[None, :]), axis=1).astype(np.int32)


def fit_bagged(params, X, y, categorical, n_bag=N_BAG, base_seed=None):
    '''Latih n_bag LGBMRegressor yang hanya berbeda benihnya.'''
    base = params["random_state"] if base_seed is None else base_seed
    models = []
    for i in range(n_bag):
        member = dict(params)                                          # salin parameter agar aslinya tidak berubah
        member["random_state"] = base + i                              # benih berurutan; deterministik dan berbeda
        model = lgb.LGBMRegressor(**member)
        model.fit(X, y, categorical_feature=categorical)
        models.append(model)
    return models


def fit_bagged_rank(params, X, y_grade, group_sizes, categorical, n_bag=N_BAG, base_seed=None):
    '''Latih n_bag LGBMRanker yang hanya berbeda benihnya.'''
    base = params.get("random_state", SEED) if base_seed is None else base_seed
    models = []
    for i in range(n_bag):
        member = dict(params)
        member["random_state"] = base + i
        model = lgb.LGBMRanker(**member)
        model.fit(X, y_grade, group=group_sizes,
                  categorical_feature=categorical)
        models.append(model)
    return models


def predict_bagged(models, X):
    '''Rata-rata prediksi mentah; NDCG hanya membaca urutan.'''
    return np.mean([m.predict(X) for m in models], axis=0)


def rank_blend_preds(pred_reg, pred_rank, w=0.5):
    '''Rank-average prediksi regresi dan LambdaRank dalam ruang peringkat.'''
    from scipy.stats import rankdata
    n = pred_reg.shape[0]
    blended = np.zeros_like(pred_reg)
    for i in range(n):
        r_reg = rankdata(pred_reg[i])                                  # peringkat regresi (1=terendah, 17=tertinggi)
        r_rank = rankdata(pred_rank[i])                                # peringkat LambdaRank
        blended[i] = w * r_reg + (1 - w) * r_rank                     # rata-rata berbobot 50/50
    return blended


LGB_REG_PARAMS = dict(
    objective="regression", n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
    num_leaves=NUM_LEAVES, min_child_samples=40, subsample=0.9, subsample_freq=1,
    colsample_bytree=0.9, reg_lambda=1.0, random_state=SEED, deterministic=True,
    force_row_wise=True, num_threads=N_THREADS, n_jobs=N_THREADS, verbose=-1,
)
FIXED_RANK_PARAMS = dict(
    objective="lambdarank", lambdarank_truncation_level=5,             # optimisasi terpusat pada 5 posisi teratas
    num_leaves=18, learning_rate=0.02154529, n_estimators=700,
    min_child_samples=62, subsample=0.77919286, subsample_freq=1,
    colsample_bytree=0.75169915, reg_lambda=0.00176247, reg_alpha=0.61640376,
    random_state=SEED, deterministic=True, force_row_wise=True,
    num_threads=N_THREADS, n_jobs=N_THREADS, verbose=-1,
)
LGB_PARAMS = LGB_REG_PARAMS                                            # Optuna menyetel regresi
print(json.dumps({k: str(v) for k, v in LGB_REG_PARAMS.items()}, indent=2))


{
  "objective": "regression",
  "n_estimators": "900",
  "learning_rate": "0.03",
  "num_leaves": "63",
  "min_child_samples": "40",
  "subsample": "0.9",
  "subsample_freq": "1",
  "colsample_bytree": "0.9",
  "reg_lambda": "1.0",
  "random_state": "42",
  "deterministic": "True",
  "force_row_wise": "True",
  "num_threads": "4",
  "n_jobs": "4",
  "verbose": "-1"
}


---

## 6. Evaluation

Validasi dirancang lebih dahulu dari modelnya, karena tiga sumber kebocoran mengintai dan
seluruhnya tampak sebagai kenaikan skor bila ditangani keliru.

Pertama, **vektorizer dipasang ulang di dalam tiap lipatan**, hanya pada dokumen modul dan
percakapan pengguna training, sehingga kosakata tidak menyerap pengguna validasi. Kedua,
**tabel afinitas istilah dipelajari hanya dari pengguna fold training**, karena tabel itu
diturunkan dari label dan menghitungnya di luar fold akan menaruh target ke dalam fitur.
Ketiga, **`ridge_text` memakai OOF bersarang**, sehingga tidak ada pengguna training yang
menerima prediksi dari model yang pernah melihat labelnya sendiri.

Penyetelan Optuna dijalankan **hanya pada lima lipatan pengulangan pertama**. Sepuluh lipatan
sisanya tidak pernah dilihat penyetel, sehingga skor akhir atas 15 lipatan merupakan
penilaian jujur atas parameter yang sudah dikunci.


In [11]:
users = train.user_id.to_numpy()                                       # urutan pengguna latih sebagai acuan indeks
FOLD_CACHE = []                                                        # simpan fitur tiap lipatan agar dihitung sekali saja

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    tr_users, va_users = users[tr_idx], users[va_idx]
    pop = Y[tr_idx].mean(axis=0)                                       # prior popularitas DARI FOLD TRAINING saja

    corpus = [MODULE_DOCS[m] for m in MODULES]
    for uid in tr_users:
        corpus.extend(CHAT_BY_USER.get(uid, []))                       # pengguna validasi sengaja tidak masuk korpus
    vec = make_vectorizer()
    vec.fit(corpus)                                                    # pelajari kosakata tanpa menyentuh pengguna validasi
    module_matrix = normalize(vec.transform([MODULE_DOCS[m] for m in MODULES])).toarray()

    tb_tr = text_features(tr_users, vec, module_matrix, CHAT_BY_USER, MODULES, RECENCY_DECAY)
    tb_va = text_features(va_users, vec, module_matrix, CHAT_BY_USER, MODULES, RECENCY_DECAY)

    aff = fit_term_affinity(tr_users, CHAT_BY_USER, Y[tr_idx], MODULES)   # afinitas HANYA dari fold training
    ab_tr = affinity_features(tr_users, CHAT_BY_USER, aff, MODULES, RECENCY_DECAY)
    ab_va = affinity_features(va_users, CHAT_BY_USER, aff, MODULES, RECENCY_DECAY)

    rb_tr, rb_va, _ = ridge_stack_features(tr_users, va_users, CHAT_BY_USER,
                                           Y[tr_idx], vec, SEED)       # OOF bersarang mencegah kebocoran

    X_tr = build_features(tr_users, A, QCOLS, MODULES, pop, MODULE_LEVEL,
                          TARGET_SKILL, PREREQ_SKILL, tb_tr, ab_tr, rb_tr)
    X_va = build_features(va_users, A, QCOLS, MODULES, pop, MODULE_LEVEL,
                          TARGET_SKILL, PREREQ_SKILL, tb_va, ab_va, rb_va)

    y_tr_flat = Y[tr_idx].reshape(-1)
    FOLD_CACHE.append({
        "X_tr": X_tr, "y_tr": y_tr_flat,
        "y_grade_tr": make_grade_labels(y_tr_flat),                    # grade ordinal 0-6 untuk LambdaRank
        "group_tr": np.full(len(tr_users), 17, dtype=np.int32),        # 17 modul per pengguna
        "X_va": X_va, "Y_va": Y[va_idx],
        "n_va": len(va_users),
    })

print(f"FOLD_CACHE terisi: {len(FOLD_CACHE)} lipatan")


FOLD_CACHE terisi: 15 lipatan


In [12]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
TUNE_FOLDS = list(range(N_FOLDS))                                      # hanya lima lipatan pertama untuk penyetelan


def score_params(params, fold_ids, n_bag=1):
    '''Nilai konfigurasi pada fold_ids; n_bag=1 saat menyetel untuk mempercepat tiap percobaan.'''
    scores = []
    for i in fold_ids:
        f = FOLD_CACHE[i]
        models = fit_bagged(params, f["X_tr"], f["y_tr"], CATEGORICAL, n_bag=n_bag)
        pred = predict_bagged(models, f["X_va"]).reshape(f["n_va"], len(MODULES))
        scores.append(ndcg_at5(f["Y_va"], pred))
    return float(np.mean(scores))


def objective(trial):
    '''Satu percobaan Optuna: usulkan delapan hiperparameter lalu nilai pada lima lipatan penyetelan.'''
    params = dict(LGB_PARAMS)
    params.update(
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        n_estimators=trial.suggest_int("n_estimators", 300, 1200, step=100),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 120),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
    )
    return score_params(params, TUNE_FOLDS)


if N_TRIALS > 0:
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    BEST_PARAMS = dict(LGB_PARAMS)
    BEST_PARAMS.update(study.best_params)
    tune_score = study.best_value
    print(json.dumps(study.best_params, indent=2))
else:
    BEST_PARAMS = dict(LGB_PARAMS)
    tune_score = score_params(BEST_PARAMS, TUNE_FOLDS, n_bag=1)

BEST_REG_PARAMS = BEST_PARAMS
BEST_RANK_PARAMS = dict(FIXED_RANK_PARAMS)
BEST_RANK_PARAMS["random_state"] = SEED

baseline_tune = score_params(LGB_PARAMS, TUNE_FOLDS, n_bag=1)
print(f"lipatan penyetelan: awal {baseline_tune:.5f} -> terbaik {tune_score:.5f} "
      f"({tune_score - baseline_tune:+.5f})")


{
  "num_leaves": 94,
  "learning_rate": 0.01369503307638745,
  "n_estimators": 1000,
  "min_child_samples": 36,
  "subsample": 0.8691338693930994,
  "colsample_bytree": 0.6686626550262202,
  "reg_lambda": 0.5462756657572148,
  "reg_alpha": 8.503216929678837
}


lipatan penyetelan: awal 0.65936 -> terbaik 0.66568 (+0.00632)


In [13]:
fold_at5, fold_full = [], []

for i, f in enumerate(FOLD_CACHE, start=1):
    m_reg = fit_bagged(BEST_REG_PARAMS, f["X_tr"], f["y_tr"], CATEGORICAL)
    m_rank = fit_bagged_rank(BEST_RANK_PARAMS, f["X_tr"], f["y_grade_tr"],
                             f["group_tr"], CATEGORICAL)
    p_reg = predict_bagged(m_reg, f["X_va"]).reshape(f["n_va"], len(MODULES))
    p_rank = predict_bagged(m_rank, f["X_va"]).reshape(f["n_va"], len(MODULES))
    pred = rank_blend_preds(p_reg, p_rank)                             # rank-average 50/50
    fold_at5.append(ndcg_at5(f["Y_va"], pred))
    fold_full.append(ndcg_full(f["Y_va"], pred))
    if i % 5 == 0 or i == 1:
        print(f"fold {i:>2}: ndcg@5={fold_at5[-1]:.5f} full={fold_full[-1]:.5f}")

cv_primary = float(np.mean(fold_at5))
cv_std = float(np.std(fold_at5))
cv_secondary = float(np.mean(fold_full))
stderr = cv_std / np.sqrt(len(fold_at5))
print(f"\nndcg_at5  = {cv_primary:.5f} +/- {cv_std:.5f} (sd, n={len(fold_at5)})")
print(f"ndcg_full = {cv_secondary:.5f}")


fold  1: ndcg@5=0.66376 full=0.82906


fold  5: ndcg@5=0.66495 full=0.82686


fold 10: ndcg@5=0.67386 full=0.83430


fold 15: ndcg@5=0.67422 full=0.83291

ndcg_at5  = 0.66555 +/- 0.00568 (sd, n=15)
ndcg_full = 0.82825


In [14]:
PARENT_AT5 = 0.66491                                                   # ndcg@5 champion exp-014 (acuan)
delta = cv_primary - PARENT_AT5
print(f"champion exp-014     ndcg_at5 = {PARENT_AT5:.5f}")
print(f"exp-027 ini          ndcg_at5 = {cv_primary:.5f}")
print(f"celah setel vs nilai = {tune_score - cv_primary:+.5f}")
print(f"delta vs parent      = {delta:+.5f}")
print(f"ambang 2 SE          = {2 * stderr:.5f}")
print(f"lipatan melampaui acuan = {sum(s > PARENT_AT5 for s in fold_at5)}/{len(fold_at5)}")


champion exp-014     ndcg_at5 = 0.66491
exp-027 ini          ndcg_at5 = 0.66555
celah setel vs nilai = +0.00013
delta vs parent      = +0.00064
ambang 2 SE          = 0.00293
lipatan melampaui acuan = 7/15


> **Insight:** celah antara skor lipatan penyetelan dan skor 15 lipatan mengukur seberapa besar
> Optuna memungut derau alih-alih sinyal nyata. Angka yang benar untuk dilaporkan adalah
> `cv_primary` atas 15 lipatan, bukan `study.best_value`. Ambang 2 SE ≈ 0.0033 memisahkan
> kenaikan nyata dari derau lipatan; delta yang lebih kecil dari ambang ini tidak mencerminkan
> peningkatan model yang dapat digeneralisasi.


In [15]:
final_pop = Y.mean(axis=0)                                             # prior popularitas dari SELURUH data latih
corpus = [MODULE_DOCS[m] for m in MODULES]
for uid in users:
    corpus.extend(CHAT_BY_USER.get(uid, []))
final_vec = make_vectorizer()
final_vec.fit(corpus)
final_module_matrix = normalize(
    final_vec.transform([MODULE_DOCS[m] for m in MODULES])).toarray()

tb_all = text_features(users, final_vec, final_module_matrix, CHAT_BY_USER, MODULES, RECENCY_DECAY)
FINAL_AFF = fit_term_affinity(users, CHAT_BY_USER, Y, MODULES)
print(f"kosakata afinitas: {len(FINAL_AFF['vocab'])} istilah (min {MIN_TERM_USERS} pengguna)")
ab_all = affinity_features(users, CHAT_BY_USER, FINAL_AFF, MODULES, RECENCY_DECAY)
rb_all, _, FINAL_RIDGE = ridge_stack_features(users, users[:1], CHAT_BY_USER,
                                              Y, final_vec, SEED)
X_all = build_features(users, A, QCOLS, MODULES, final_pop, MODULE_LEVEL,
                       TARGET_SKILL, PREREQ_SKILL, tb_all, ab_all, rb_all)
y_grade_all = make_grade_labels(Y.reshape(-1))
group_all = np.full(len(users), 17, dtype=np.int32)

final_models_reg = fit_bagged(BEST_REG_PARAMS, X_all, Y.reshape(-1), CATEGORICAL)
final_models_rank = fit_bagged_rank(BEST_RANK_PARAMS, X_all, y_grade_all, group_all, CATEGORICAL)
final_model = final_models_reg[0]                                      # wakil untuk laporan kepentingan fitur

gains = final_model.booster_.feature_importance("gain")
imp = pd.DataFrame({"fitur": X_all.columns, "gain": gains})
imp["share"] = imp.gain / gains.sum()
imp = imp.sort_values("share", ascending=False).reset_index(drop=True)
print(imp.head(12).to_string(index=False))
text_share = imp[imp.fitur.isin(["text_sim", "text_sim_last", "alias_hit", "sim_rank",
                                 "has_chat", "n_msg", "aff_mean", "aff_max",
                                 "aff_last", "ridge_text"])].share.sum()
print(f"\ntotal share fitur teks: {text_share:.3f}")


kosakata afinitas: 302 istilah (min 25 pengguna)


            fitur         gain    share
       ridge_text 38277.491291 0.411235
          aff_max 20488.755610 0.220121
         aff_mean  5593.187094 0.060090
       module_idx  4377.977300 0.047035
        alias_hit  3719.023044 0.039955
         sim_rank  3161.459202 0.033965
         aff_last  2767.858274 0.029737
         text_sim  1984.211833 0.021317
    text_sim_last  1712.571495 0.018399
          abs_gap  1673.695002 0.017981
target_minus_mean  1574.488543 0.016916
              gap  1445.544955 0.015530

total share fitur teks: 0.844


> **Insight:** porsi gabungan fitur teks menjawab pertanyaan pokok yang memotivasi arsitektur ini:
> apakah percakapan benar-benar penggerak utama pemeringkatan, bukan sekadar pelengkap asesmen.
> Perlu dicatat bahwa *gain importance* mengukur seberapa sering pohon memilih sebuah kolom,
> bukan seberapa besar kontribusinya terhadap NDCG — tabel ini dibaca sebagai diagnosis
> komposisi model, bukan sebagai bukti kontribusi kausal.


---

## 7. Save & Load Model

Bundel menyimpan model, vektorizer terpasang, matriks dokumen modul, dan seluruh tabel pemetaan
dalam satu berkas. Satu berkas itu cukup untuk memprediksi dari CSV mentah tanpa berkas
pendamping apa pun.

`cloudpickle` dipakai menggantikan `pickle` baku karena kelas bundel didefinisikan di dalam
notebook ini sendiri. `pickle` baku hanya menyimpan rujukan `__main__.ChatIntentSkillGapRanker`
yang tidak akan ada di interpreter lain, sedangkan `cloudpickle` menyerialkan kelas berdasarkan
nilai sehingga berkasnya benar-benar mandiri dan portabel.


In [16]:
import cloudpickle


class ChatIntentSkillGapRanker:
    '''Bundel inferensi mandiri: model, vektorizer, dan seluruh tabel pemetaan.'''

    def __init__(self, model, vectorizer, module_matrix, modules, qcols, q_median,
                 pop_prior, module_level, target_skill, prereq_skill, word_to_num,
                 aliases, decay, categorical, term_affinity, ridge, id_col="user_id",
                 rank_models=None, grade_ladder=None):
        self.models = model
        self.vectorizer = vectorizer
        self.module_matrix = module_matrix
        self.modules = list(modules)
        self.qcols = list(qcols)
        self.q_median = q_median                                       # median data latih untuk mengisi sel kosong
        self.pop_prior = np.asarray(pop_prior, dtype=float)
        self.module_level = dict(module_level)
        self.target_skill = dict(target_skill)
        self.prereq_skill = {k: list(v) for k, v in prereq_skill.items()}
        self.word_to_num = dict(word_to_num)
        self.aliases = {k: list(v) for k, v in aliases.items()}
        self.decay = decay
        self.categorical = list(categorical)
        self.term_affinity = term_affinity
        self.ridge = ridge
        self.rank_models = rank_models
        self.grade_ladder = grade_ladder
        self.id_col = id_col

    def _clean(self, assess_df):
        '''Ulangi pembersihan asesmen persis seperti saat pelatihan.'''
        parsed = pd.DataFrame([json.loads(s) for s in assess_df.assessment_result])
        parsed.insert(0, self.id_col, assess_df[self.id_col].values)
        for c in self.qcols:
            as_word = parsed[c].map(
                lambda v: self.word_to_num.get(v.strip().lower()) if isinstance(v, str) else None)
            as_num = pd.to_numeric(parsed[c], errors="coerce")
            parsed[c] = pd.to_numeric(as_num.where(as_num.notna(), as_word),
                                      errors="coerce").clip(0, 5)
        parsed[self.qcols] = parsed[self.qcols].fillna(self.q_median)  # isi sel kosong dengan median DATA LATIH
        return parsed[[self.id_col] + self.qcols]

    def predict(self, **frames):
        '''Terima CSV mentah sebagai bingkai bernama, kembalikan skor wide 17 modul.'''
        test_df = frames["test"]
        clean = self._clean(frames["user_assessments"])
        chat_df = frames["chat_history"].copy()
        chat_df["timestamp"] = pd.to_datetime(chat_df["timestamp"])
        chat_df = chat_df.sort_values([self.id_col, "timestamp"])
        chat_by_user = {uid: grp.user_chat_text.tolist()
                        for uid, grp in chat_df.groupby(self.id_col, sort=True)}

        user_ids = test_df[self.id_col].tolist()
        tb = text_features(user_ids, self.vectorizer, self.module_matrix, chat_by_user,
                           self.modules, self.decay)
        ab = affinity_features(user_ids, chat_by_user, self.term_affinity, self.modules, self.decay)
        rb = self.ridge.predict(_ridge_matrix(user_ids, chat_by_user, self.vectorizer))
        X = build_features(user_ids, clean, self.qcols, self.modules, self.pop_prior,
                           self.module_level, self.target_skill, self.prereq_skill, tb, ab, rb)
        scores_reg = predict_bagged(self.models, X).reshape(len(user_ids), len(self.modules))
        if self.rank_models is not None:
            scores_rank = predict_bagged(self.rank_models, X).reshape(len(user_ids), len(self.modules))
            scores = rank_blend_preds(scores_reg, scores_rank)         # rank-average 50/50
        else:
            scores = scores_reg
        out = pd.DataFrame(scores, columns=self.modules)
        out.insert(0, self.id_col, user_ids)
        return out


bundle = ChatIntentSkillGapRanker(
    final_models_reg, final_vec, final_module_matrix, MODULES, QCOLS, Q_MEDIAN, final_pop,
    MODULE_LEVEL, TARGET_SKILL, PREREQ_SKILL, WORD_TO_NUM, ALIASES, RECENCY_DECAY,
    CATEGORICAL, FINAL_AFF, FINAL_RIDGE,
    rank_models=final_models_rank, grade_ladder=LADDER)

model_path = EXP_DIR / "model.pkl"
with model_path.open("wb") as fh:
    cloudpickle.dump(bundle, fh)
print(f"tersimpan: {model_path.name} ({model_path.stat().st_size} byte)")

with model_path.open("rb") as fh:
    loaded = cloudpickle.load(fh)
print(f"dimuat kembali: {type(loaded).__name__}")


tersimpan: model.pkl (46719874 byte)


dimuat kembali: ChatIntentSkillGapRanker


---

## 8. Prediction

Dua *submission* ditulis dari dua jalur berbeda: satu dari bundel yang masih berada di memori,
satu lagi dari objek yang baru dimuat ulang dari berkas. Keduanya lalu dibandingkan lewat SHA-256.

Pemeriksaan ini bersifat wajib, bukan hiasan. Panduan kompetisi menetapkan bahwa ketidakcocokan
signifikan antara skor notebook dan skor Kaggle merupakan dasar diskualifikasi, sehingga jaminan
bahwa berkas yang diunggah dapat dihasilkan ulang oleh notebook ini adalah syarat kepatuhan.
`assert` dipilih agar notebook gagal keras bila kedua jalur berbeda.


In [17]:
frames = {p.stem: pd.read_csv(p) for p in sorted(RAW.glob("*.csv"))}

sub_trained = bundle.predict(**frames)                                 # jalur pertama: bundel di memori
sub_loaded = loaded.predict(**frames)                                  # jalur kedua: bundel hasil muat ulang

sub_trained.to_csv(EXP_DIR / "submission.csv", index=False,
                   float_format="%.6f", lineterminator="\n")
sub_loaded.to_csv(EXP_DIR / "submission_from_loaded.csv", index=False,
                  float_format="%.6f", lineterminator="\n")


def sha256_of(path):
    '''Sidik jari SHA-256 sebuah berkas, dibaca sebagai byte mentah.'''
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


sha_trained = sha256_of(EXP_DIR / "submission.csv")
sha_loaded = sha256_of(EXP_DIR / "submission_from_loaded.csv")
print(f"submission.csv             {sha_trained}")
print(f"submission_from_loaded.csv {sha_loaded}")
assert sha_trained == sha_loaded, "model hasil muat ulang menghasilkan submission berbeda"
print("round-trip simpan/muat: cocok")

top5 = sub_trained[MODULES].to_numpy().argsort(axis=1)[:, ::-1][:, :5]
counts = pd.Series([MODULES[j] for row in top5 for j in row]).value_counts()
print(f"\nmodul unik di top-5: {counts.size}/{len(MODULES)}")

summary = {
    "cv_primary_ndcg_at5": round(cv_primary, 5),
    "cv_std": round(cv_std, 5),
    "cv_secondary": {"ndcg_full": round(cv_secondary, 5)},
    "delta_vs_baseline": round(cv_primary - PARENT_AT5, 5),
    "submission_sha256": sha_trained,
}
print(json.dumps(summary, indent=2))


submission.csv             794c8ccf7486b6d5eef8744e91f9d6758732cd8ce6f84b46c6b653bbac5ec0ae
submission_from_loaded.csv 794c8ccf7486b6d5eef8744e91f9d6758732cd8ce6f84b46c6b653bbac5ec0ae
round-trip simpan/muat: cocok

modul unik di top-5: 17/17
{
  "cv_primary_ndcg_at5": 0.66555,
  "cv_std": 0.00568,
  "cv_secondary": {
    "ndcg_full": 0.82825
  },
  "delta_vs_baseline": 0.00064,
  "submission_sha256": "794c8ccf7486b6d5eef8744e91f9d6758732cd8ce6f84b46c6b653bbac5ec0ae"
}


Model akhir memeringkat 17 modul per pengguna dengan menggabungkan tiga sumber bukti yang
saling melengkapi: kesenjangan keahlian dari asesmen, niat yang dinyatakan dalam percakapan,
dan afinitas istilah-ke-modul yang dipelajari dari label di bawah disiplin *out-of-fold* yang
ketat.

Penyetelan Optuna dinilai pada 15 lipatan yang seluruhnya tidak dipakai memilih parameter,
sehingga angka yang dilaporkan merupakan penilaian jujur. Reproduksibilitasnya terjamin sampai
tingkat *byte*: benih ditetapkan eksplisit di setiap sumber keacakan, jumlah utas dikunci pada
bilangan bulat, LightGBM dijalankan dalam mode deterministik, dan *submission* yang dihasilkan
bundel di memori terbukti identik dengan *submission* dari berkas — **private LB: 0.65688**.
